# Ask Coresight v5: the consolidated training notebook (about 80 minutes on an A100, ~15-20 compute units)

**What the analysis on 13 Nov 2026 found, and what each step here does about it**

| Measured | Step in this notebook |
|---|---|
| Of 17 golden questions still wrong after the AI check, the right survey question was found 1st or 2nd in **8**: the final choice went wrong | 5-6: a reranker that reads the question with each candidate and decides, taught by a large teacher |
| In **8 more**, it was found 3rd to 10th: past the 5 that v4's reranker reads | 5-7: the reranker reads the nearest **20**, and the search model is trained for *finding* (right one in the top 20), not for being first |
| gpt-4o-mini (the AI check) itself picked wrong in 5 of the 17 where the right one was first | 3: practice questions a large model doubts are removed before training (round-trip check), so the student does not learn the teacher's mistakes |
| v4's reranker learns only *order*, so its scores say nothing about "none of these" | 6: it also learns yes/no per candidate, and off-topic questions where every candidate is a no: its score can then say "not in the surveys" |
| Headlines read "24.8% answered 'About the same' in 19 Oct 2026 – 9 Nov 2026" | 2: the large model writes, once per survey question, a chart title and a headline pattern ("{share} bought beauty products from {answer}…") |
| v4's student copied its teacher (70.9% vs the teacher's 69.5%), mistakes included | 6: it learns mostly from the known right answer (80%), the teacher's order only 20% |
| Some "wrong" candidates are as right as the right one (the same thing asked another way) | 6: a candidate the teacher rates at 95% of the right one or more is left out of the "no" lesson (NVIDIA's NV-Retriever) |
| Qwen3 and Gemini Embedding average checkpoints saved during training ("model soup") | 6-7: the end of each epoch is saved and averaged; the better of the two on held-out wording is kept |
| v4 inside Ask never reached 98% right at any threshold; today's model answers 85 of 161 at once, 98.8% right | 8: the bar v5 must beat in Ask before it ships |

**How to run:** Runtime → Change runtime type → **A100** (an **L4** works too, about twice as slow) → Save. Runtime → **Run all**. At **Choose Files** pick, from
`Downloads/cip-colab`, `catalogue.json` and `synthetic.json`, and if you have them `synthetic-v4.json` (from v4: saves 25
minutes), `qwen-v5.json` (from an earlier v5 run: saves 30 minutes) and `learned.json` (staging's learned answers). Keep the
laptop open. `synthetic-v4.json` (if written) and `qwen-v5.json` download on the way (keep both: a re-run that uploads them skips
steps 2-3), `ask-v5.zip` at the end; the GPU is released 15 minutes later.

**Monthly:** run this same notebook again with staging's newest `learned.json`: real questions the AI settled become
training data. That is the whole retraining plan; no new notebook versions.

In [ ]:
# quiet: Colab's own packages are not used here, so pip's notes about them are hidden
!pip -q install "sentence-transformers==3.3.1" accelerate onnx onnxscript onnxruntime > /dev/null 2>&1   # 3.3.1: trained here on 12 Nov
import torch, json, random, time, os, re, shutil, numpy as np
from collections import defaultdict
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → GPU")
DEVICE = "cuda"

In [ ]:
from google.colab import files
if not (os.path.exists("catalogue.json") and os.path.exists("synthetic.json")):
    print("Pick catalogue.json and synthetic.json, and any of synthetic-v4.json, qwen-v5.json, learned.json you have (hold ⌘)")
    files.upload()
data = json.load(open("catalogue.json"))
docs, golden, golden_none = data["docs"], data["golden"], data.get("golden_none", [])
TRAIN_ASKS = 6   # asks 1-6 train; asks 7-8, the golden questions and the golden "not in the surveys" are the test, never trained on
at = {d["key"]: n for n, d in enumerate(docs)}
v3 = json.load(open("synthetic.json"))
v4 = json.load(open("synthetic-v4.json")) if os.path.exists("synthetic-v4.json") else None
learned = json.load(open("learned.json")) if os.path.exists("learned.json") else []
held_out = {g["q"].lower() for g in golden} | {q.lower() for q in golden_none}
REAL = [(v["question"], at[v["key"]]) for v in learned if v.get("key") in at and v["question"].lower() not in held_out]
for d in docs:
    d["v3"] = v3.get(d["key"], [])
    d["answers"] = d["embed"].split("| answers: ")[-1].split("; ") if "| answers: " in d["embed"] else []
print(len(docs), "survey questions ·", len(golden), "golden ·", len(golden_none), "golden not in the surveys ·",
      sum(len(d["v3"]) for d in docs), "v3 practice questions ·", "v4 practice questions found" if v4 else "no synthetic-v4.json: step 3 writes them",
      "·", len(REAL), "real questions from staging")

In [ ]:
# 1. Today's search: how often is the right survey question first, and how often is it among the nearest 5, 10, 20?
# (a reranker can only choose what the search found: being in the top 20 is what matters most)
from sentence_transformers import SentenceTransformer

def index_texts():
    owners, texts = [], []
    for n, d in enumerate(docs):
        for t in [d["embed"]] + d["asks"][:TRAIN_ASKS]:
            owners.append(n); texts.append(t)
    return np.array(owners), texts

def ranked(encode, queries, k=20):
    # for each query: its k nearest survey questions (by their best text) and their closeness, in chunks
    owners, texts = index_texts()
    vec = encode(texts); out = []
    for s in range(0, len(queries), 2048):
        for row in encode(queries[s:s + 2048]) @ vec.T:
            per = np.full(len(docs), -1.0); np.maximum.at(per, owners, row)
            top = np.argsort(-per)[:k]; out.append((top, per[top]))
    return out

def st_encoder(model):
    return lambda texts: model.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

TESTS = {"unseen": ([a for d in docs for a in d["asks"][TRAIN_ASKS:]], [n for n, d in enumerate(docs) for _ in d["asks"][TRAIN_ASKS:]]),
         "golden": ([g["q"] for g in golden], golden)}
def right(name, top, want):
    return docs[top]["group"] == docs[want]["group"] if name == "unseen" else any(w in docs[top]["text"].lower() for w in want["want"])

def found_at(label, encode):
    out = {}
    for name, (queries, wants) in TESTS.items():
        near = ranked(encode, queries)
        out[name] = {k: float(np.mean([any(right(name, c, w) for c in top[:k]) for (top, _), w in zip(near, wants)])) for k in (1, 5, 10, 20)}
        print(f"{label}, {name}: " + " · ".join(f"in top {k}: {v:.1%}" for k, v in out[name].items()))
    return out

base = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
results = {"search today": found_at("today's search", st_encoder(base))}

In [ ]:
# 2-3. Qwen2.5-7B-Instruct (Apache 2.0), once: practice questions (if v4's are not uploaded), off-topic questions, a chart
# title and headline pattern for every survey question, and the round-trip check of practice questions
WRITE_V4, WRITE_QWEN = v4 is None, not os.path.exists("qwen-v5.json")
NEED_QWEN = WRITE_V4 or WRITE_QWEN
if NEED_QWEN:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="cuda")

    def chat(system, user):
        return tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)

    TOKENS = 24000 if torch.cuda.get_device_properties(0).total_memory > 30e9 else 10000   # an A100 holds ~2.5x an L4 (22 GB)
    def generate(jobs, max_new, temperature=0.7):
        # jobs: [(tag, prompt)] → [(tag, text)]; prompts of similar length together, as many as the GPU holds; if it still
        # runs out of memory, half as many and the same prompts again (an L4 stopped at 1,056 of 3,076, 13 Nov 2026)
        global TOKENS
        order = sorted(range(len(jobs)), key=lambda i: len(jobs[i][1])); out = [None] * len(jobs); started = time.time(); s = 0; rounds = 0
        while s < len(order):
            longest = len(tok(jobs[order[min(s + 95, len(order) - 1)]][1]).input_ids) + max_new
            part = order[s:s + max(1, min(96, TOKENS // longest))]
            try:
                enc = tok([jobs[i][1] for i in part], return_tensors="pt", padding=True).to("cuda")
                with torch.no_grad():
                    gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=True, temperature=temperature, top_p=0.9, pad_token_id=tok.eos_token_id)
            except torch.cuda.OutOfMemoryError:
                enc = gen = None; torch.cuda.empty_cache(); TOKENS //= 2
                print(f"(GPU memory full: {TOKENS} tokens a batch from now on)", end=" | ")
                continue
            for i, seq in zip(part, gen[:, enc["input_ids"].shape[1]:]):
                out[i] = (jobs[i][0], tok.decode(seq, skip_special_tokens=True))
            s += len(part); rounds += 1
            if rounds % 10 == 0:
                print(f"{s} of {len(jobs)} · {time.time() - started:.0f}s", end=" | ")
        print(f"{len(jobs)} of {len(jobs)} · {time.time() - started:.0f}s")
        return out

    def json_in(text, opening="["):
        try:
            return json.loads(text[text.index(opening):text.rindex("]" if opening == "[" else "}") + 1])
        except ValueError:
            return None

    def options(d):
        return "; ".join(a[:40] for a in d["answers"][:8])

In [ ]:
# 2a. v4's practice questions, written again only when synthetic-v4.json was not uploaded (~25 minutes)
if WRITE_V4:
    KINDS = ("You write what people type into a consumer-survey search box. For the survey question below, write 16 short questions "
             "someone would type to get THIS question's figures, one of each kind, in this order: which/what are the top ones; "
             "what share or how many people; yes-or-no about people (do/are/have people…); why or the reasons (only if it asks why, "
             "else another share question); how much or what amount (only if it asks an amount, else another which question); "
             "over time (trend, since a year, by month or quarter); by generation; by income or region; men versus women; "
             "comparing two years or seasons; about one of its answer options by name; a retail client's decision; an analyst's "
             "precise wording; a colleague's casual wording; a casual wording with a typo; a very short one (3-5 words). Keep what "
             "kind of answer the survey question gives (a list, yes/no, a reason, an amount): never ask something another kind of "
             "question would answer. Never copy the survey wording, no numbers from the answers. Reply with a JSON list of 16 strings only.")
    CONTRAST = ("Two consumer-survey questions look alike but ask different things. Write 6 short questions someone would type that "
                "QUESTION A answers and QUESTION B does not (use what makes A different: a list vs yes/no, a reason, an amount, the "
                "event, the period, the retailer). Plain words, never copy the survey wording. Reply with a JSON list of 6 strings only.")
    near = ranked(st_encoder(base), [d["embed"] for d in docs], k=8)
    jobs = [(d["key"], chat(KINDS, f"Survey question: {d['text'][:300]}\nAnswer options: {options(d)}")) for d in docs]
    for n, d in enumerate(docs):
        s = next((int(o) for o in near[n][0] if docs[o]["group"] != d["group"]), None)
        if s is not None:
            jobs.append((d["key"], chat(CONTRAST, f"QUESTION A: {d['text'][:250]}\nA's answers: {options(d)}\n"
                                                   f"QUESTION B: {docs[s]['text'][:250]}\nB's answers: {options(docs[s])}")))
    v4 = {}
    for key, text in generate(jobs, 480):
        asks = json_in(text) or re.findall(r'"([^"]{6,160})"', text)
        v4.setdefault(key, []).extend([a.strip() for a in asks if isinstance(a, str) and 6 < len(a.strip()) < 160][:16])
    json.dump(v4, open("synthetic-v4.json", "w"))
    files.download("synthetic-v4.json")   # keep it: a re-run uploads it and skips this step
for d in docs:
    d["v4"] = v4.get(d["key"], []) if v4 else []
print("v4 practice questions:", sum(len(d["v4"]) for d in docs))

In [ ]:
# 2b. Off-topic questions (the surveys cannot answer them): teach the reranker what "none of these" looks like.
# 2c. Every survey question's chart title and headline pattern, checked before it is kept (the server falls back to
# today's wording for any question without one)
TYPES = {"list", "single", "scale", "yesno", "amount"}
def valid(p, d):
    if not (isinstance(p, dict) and isinstance(p.get("title"), str) and isinstance(p.get("finding"), str) and p.get("type") in TYPES):
        return None
    title, finding = p["title"].strip().rstrip("."), p["finding"].strip().rstrip(".")
    if not (3 <= len(title) <= 70 and finding.count("{share}") == 1 and finding.count("{answer}") <= 1 and len(finding) <= 140
            and not re.search(r"\d", finding) and not re.search(r"\{(?!share\}|answer\})", finding)):
        return None
    answers = set(d["answers"])
    return {"title": title, "finding": finding, "type": p["type"],
            "positive": [a for a in p.get("positive") or [] if a in answers], "negative": [a for a in p.get("negative") or [] if a in answers]}

if WRITE_QWEN:
    OFFTOPIC = ("Write 25 short questions a person might type into a US consumer-survey search box that NO survey of shoppers' "
                "opinions and habits can answer: facts from elsewhere such as stock prices, a company's revenue or store count, who "
                "runs a company, weather, sports results, news events, product specifications, store opening hours, recipes, "
                "translations, how-to questions. Vary the wording: some casual, some naming retailers or brands. Reply with a JSON list of 25 strings only.")
    offtopic = sorted({a.strip() for _, text in generate([("x", chat(OFFTOPIC, f"Batch {i}: make these different from any other batch."))
                                                          for i in range(90)], 700, temperature=0.95)
                       for a in (json_in(text) or []) if isinstance(a, str) and 6 < len(a.strip()) < 160})
    PHRASE = ("You write chart titles and headline templates for a consumer-survey research report. For the survey question and "
              "its answer options, reply with JSON only: {\"title\": a chart title of at most 8 words naming what is measured, "
              "sentence case, no period; \"type\": \"list\" (choose several), \"single\" (choose one), \"scale\" (an ordered scale "
              "such as better/worse or agree/disagree), \"yesno\" or \"amount\"; \"finding\": a headline template in plain English "
              "with {share} for the percentage and {answer} for one answer option, for example \"{share} bought groceries from "
              "{answer} in the past two weeks\" or \"{share} expect the economy to be {answer} over the next 12 months\" (for a "
              "yes/no question it describes the yes answer and may leave out {answer}); \"positive\": the answer options that count "
              "as positive on a scale, else []; \"negative\": those that count as negative, else []}. No numbers in the template.")
    phrases, tries = {}, 0
    todo = list(range(len(docs)))
    while todo and tries < 3:   # a pattern that fails the checks is asked again, up to three times
        made = generate([(n, chat(PHRASE, f"Survey question: {docs[n]['text'][:300]}\nAnswer options: {'; '.join(a[:40] for a in docs[n]['answers'][:20])}"))
                         for n in todo], 220, temperature=0.3 + 0.2 * tries)
        for n, text in made:
            p = valid(json_in(text, "{"), docs[n])
            if p:
                phrases[docs[n]["key"]] = p
        todo = [n for n in todo if docs[n]["key"] not in phrases]; tries += 1
    print(len(offtopic), "off-topic questions ·", len(phrases), "of", len(docs), "survey questions have a checked title and headline pattern")
    for d in random.Random(3).sample(docs, 4):
        print("·", d["text"][:80], "→", phrases.get(d["key"]))

In [ ]:
# 3. The round-trip check (as in Promptagator and InPars): a practice question written for survey question A is kept only
# if a large model, shown A among its four closest different questions, also picks A. Only the hard ones are checked (the
# search does not already put A first); its own accuracy on the golden questions is printed so it can be trusted or not
JUDGE_MAX = 25000
if WRITE_QWEN:
    JUDGE = ("You match a person's question to the consumer-survey question whose figures answer it. Reply with the number of the "
             "ONE survey question that answers it best, or 0 if none of them does. Reply with the number only.")
    digits = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(6)]
    def judge(items):
        # items: [(question, [doc indices])] → probabilities over 0 (none) and each candidate in order
        prompts = [chat(JUDGE, "Survey questions:\n" + "\n".join(f"{i + 1}. {docs[c]['text'][:160]}" for i, c in enumerate(cands))
                        + f"\n0. None of these\nPerson's question: {q}") for q, cands in items]
        out, started, size, s = [], time.time(), 32, 0
        while s < len(prompts):
            try:
                enc = tok(prompts[s:s + size], return_tensors="pt", padding=True).to("cuda")
                with torch.no_grad():
                    last = llm.model(**enc, position_ids=(enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)).last_hidden_state[:, -1]
                    out.extend(torch.softmax(llm.lm_head(last)[:, digits].float(), -1).cpu().numpy())
            except torch.cuda.OutOfMemoryError:   # half as many at once, the same prompts again
                enc = last = None; torch.cuda.empty_cache(); size = max(1, size // 2); continue
            s += size
            if s // size % 100 == 0:
                print(f"{min(s, len(prompts))} of {len(prompts)} · {time.time() - started:.0f}s", end=" | ")
        print()
        return out
    def distinct(top, k, keep=None):
        chosen, groups = ([keep], {docs[keep]["group"]}) if keep is not None else ([], set())
        for c in top:
            if len(chosen) < k and docs[int(c)]["group"] not in groups:
                chosen.append(int(c)); groups.add(docs[int(c)]["group"])
        return chosen
    # the judge on the golden questions whose right answer is among the five shown
    near_g = ranked(st_encoder(base), [g["q"] for g in golden])
    items = [(g["q"], distinct(top, 5)) for g, (top, _) in zip(golden, near_g)]
    seen = [(i, p) for i, ((q, cands), p) in enumerate(zip(items, judge(items))) if any(right("golden", c, golden[i]) for c in cands)]
    judge_golden = float(np.mean([int(np.argmax(p)) > 0 and right("golden", items[i][1][int(np.argmax(p)) - 1], golden[i]) for i, p in seen]))
    print(f"the judge picks the right one for {judge_golden:.1%} of {len(seen)} golden questions whose right answer it was shown"
          + ("" if judge_golden >= 0.8 else " · LOW: its rejections are kept only as a report, not applied"))
    # the practice questions the search does not already answer right
    practice = [(t, n) for n, d in enumerate(docs) for t in d["v3"] + d["v4"] if t]
    random.Random(2).shuffle(practice)
    near_p = ranked(st_encoder(base), [t for t, _ in practice], k=8)
    hard = [(t, n, top) for (t, n), (top, _) in zip(practice, near_p) if docs[int(top[0])]["group"] != docs[n]["group"]][:JUDGE_MAX]
    items, where = [], []
    for t, n, top in hard:
        cands = distinct(top, 5, keep=n)
        random.Random(len(t)).shuffle(cands)
        items.append((t, cands)); where.append(cands.index(n) + 1)
    picks = [int(np.argmax(p)) for p in judge(items)]
    rejected = [[t, docs[n]["key"]] for (t, n, _), w, p in zip(hard, where, picks) if p != w] if judge_golden >= 0.8 else []
    print(f"{len(practice)} practice questions · {len(hard)} hard ones checked · {len(rejected)} rejected ({len(rejected) / max(len(hard), 1):.0%})")
    for t, k in random.Random(4).sample(rejected, min(5, len(rejected))):
        print("  rejected:", t, "→ written for:", docs[at[k]]["text"][:80])
    json.dump({"offtopic": offtopic, "phrases": phrases, "rejected": rejected, "judge_golden": judge_golden}, open("qwen-v5.json", "w"))
    files.download("qwen-v5.json")   # keep it: a re-run uploads it (with synthetic-v4.json) and skips steps 2-3
if NEED_QWEN:
    del llm; torch.cuda.empty_cache()
qwen = json.load(open("qwen-v5.json"))
REJECTED = {(t, at[k]) for t, k in qwen["rejected"] if k in at}
offtopic = qwen["offtopic"][:]
random.Random(6).shuffle(offtopic)
OFF_TEST, OFF_TRAIN = offtopic[:300] + golden_none, offtopic[300:]
results["judge on golden"] = qwen["judge_golden"]
print(len(qwen["phrases"]), "headline patterns ·", len(REJECTED), "practice questions removed ·", len(OFF_TRAIN), "off-topic to train ·", len(OFF_TEST), "to test")

In [ ]:
# 4. The search model, trained to FIND (the right one in the nearest 20), not to be first: the reranker decides. Near-twins
# (the same question asked by other studies, one group) never appear in one batch, so none is trained as another's wrong answer
train = [(t, n) for n, d in enumerate(docs) for t in d["asks"][:TRAIN_ASKS] + [d["name"]] + d["aliases"] + d["v3"] + d["v4"]
         if t and (t, n) not in REJECTED] + REAL * 3   # a real question counts three times
by_group = defaultdict(list)
for t, n in train:
    by_group[docs[n]["group"]].append((t, n))
rng = random.Random(7)
for v in by_group.values():
    rng.shuffle(v)
batches, queues = [], [v for v in by_group.values()]
while queues:   # round robin: each round takes one from every group, so a batch never holds two of one group
    rng.shuffle(queues)
    round_ = [q.pop() for q in queues]
    batches += [round_[s:s + 128] for s in range(0, len(round_), 128) if len(round_[s:s + 128]) >= 16]
    queues = [q for q in queues if q]
rng.shuffle(batches)
search = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
opt = torch.optim.AdamW(search.parameters(), lr=2e-5)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (len(batches) // 10 + 1)) * max(0.0, 1 - s / len(batches)))
def vectors(texts):
    f = {k: v.to(DEVICE) for k, v in search.tokenize(texts).items()}
    return torch.nn.functional.normalize(search(f)["sentence_embedding"], dim=-1)
search.train(); started = time.time()
for step, batch in enumerate(batches):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        q, d = vectors([t for t, _ in batch]), vectors([docs[n]["embed"] for _, n in batch])
    loss = torch.nn.functional.cross_entropy(q.float() @ d.float().T * 20, torch.arange(len(batch), device=DEVICE))
    loss.backward(); opt.step(); sched.step(); opt.zero_grad()
    if step % 500 == 0: print(f"step {step}/{len(batches)} loss {loss.item():.3f} · {time.time() - started:.0f}s", end=" | ")
search.eval()
print(f"\n{len(train)} pairs trained in {time.time() - started:.0f}s")
results["search v5"] = found_at("v5 search", st_encoder(search))
better = (results["search v5"]["golden"][20], results["search v5"]["unseen"][20]) > (results["search today"]["golden"][20], results["search today"]["unseen"][20])
SEARCH, SEARCH_NAME = (search, "bge-cip-v5") if better else (base, "bge-small-en-v1.5")
results["search shipped"] = SEARCH_NAME
print("the reranker reads the nearest from:", SEARCH_NAME, "(v5 finds more)" if better else "(v5 does not find more: today's is kept)")

In [ ]:
# 5. The teacher (bge-reranker-v2-m3, 568M weights) reads each practice question with its nearest 20 survey questions.
# First: how good is it here, reordering the nearest 20? (the ceiling for the student)
from sentence_transformers import CrossEncoder
teacher = CrossEncoder("BAAI/bge-reranker-v2-m3", device=DEVICE, max_length=256, default_activation_function=torch.nn.Identity(),
                       automodel_args={"torch_dtype": torch.float16})
def teacher_scores(pairs):
    return teacher.predict(pairs, batch_size=256, show_progress_bar=False, convert_to_numpy=True)   # fits an L4 (22 GB) too
NEAREST = {name: ranked(st_encoder(SEARCH), queries) for name, (queries, _) in TESTS.items()}
for name, (queries, wants) in TESTS.items():
    take = range(min(len(queries), 2000 if name == "unseen" else len(queries)))
    t = [teacher_scores([(queries[i], docs[c]["embed"]) for c in NEAREST[name][i][0]]) for i in take]
    results[f"teacher {name}"] = float(np.mean([right(name, NEAREST[name][i][0][int(np.argmax(t[j]))], wants[i]) for j, i in enumerate(take)]))
    print(f"{name}: search first {np.mean([right(name, NEAREST[name][i][0][0], wants[i]) for i in take]):.1%} · teacher reordering the nearest 20: {results[f'teacher {name}']:.1%}")
POOL = 60000
pool = train[:]
random.Random(1).shuffle(pool)
pool = [(t, n) for t, n in pool[:POOL]] + [(t, None) for t in OFF_TRAIN]   # off-topic: no right answer among the 20
near = ranked(st_encoder(SEARCH), [t for t, _ in pool])
lists, started = [], time.time()
for s in range(0, len(pool), 4096):
    chunk = []
    for (q, n), (top, _) in zip(pool[s:s + 4096], near[s:s + 4096]):
        cands = [int(c) for c in top[:20]]
        if n is not None and not any(docs[c]["group"] == docs[n]["group"] for c in cands):
            cands[-1] = n   # the right one always among the 20, so the student learns to put it first
        chunk.append((q, n, cands))
    logits = teacher_scores([(q, docs[c]["embed"]) for q, _, cands in chunk for c in cands])
    at_ = 0
    for q, n, cands in chunk:
        lists.append((q, cands, logits[at_:at_ + 20].astype(np.float32),
                      np.array([n is not None and docs[c]["group"] == docs[n]["group"] for c in cands], np.float32)))
        at_ += 20
    print(f"{min(s + 4096, len(pool))} of {len(pool)} · {time.time() - started:.0f}s", end=" | ")
del teacher; torch.cuda.empty_cache()
print("\n", len(lists), "practice questions with 20 teacher-scored candidates each")

In [ ]:
# 6. The student: a small cross-encoder (MiniLM-L6, 22M weights, Apache 2.0). Two lessons at once: the ORDER the teacher
# gives the 20 (mixed with what is right), and a yes/no for each candidate, so its score also means "is this the one"
# (off-topic questions: all no). The yes/no is what lets the server say "not in the surveys" and measure how sure it is
from transformers import AutoModelForSequenceClassification, AutoTokenizer
STUDENT = "cross-encoder/ms-marco-MiniLM-L-6-v2"
stok = AutoTokenizer.from_pretrained(STUDENT)
student = AutoModelForSequenceClassification.from_pretrained(STUDENT, num_labels=1).to(DEVICE)
def student_logits(model, pairs):
    enc = stok([p[0] for p in pairs], [p[1] for p in pairs], truncation=True, max_length=256, padding=True, return_tensors="pt").to(DEVICE)
    return model(**enc).logits.view(-1)
EPOCHS, QUERIES, TEMP = 2, 8, 2.0
opt = torch.optim.AdamW(student.parameters(), lr=3e-5)
steps = EPOCHS * (len(lists) // QUERIES)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (steps // 10 + 1)) * max(0.0, 1 - s / steps))
student.train(); started = time.time(); step = 0; saved = []
for epoch in range(EPOCHS):
    random.Random(epoch).shuffle(lists)
    for s in range(0, len(lists) - QUERIES + 1, QUERIES):
        part = lists[s:s + QUERIES]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = student_logits(student, [(q, docs[c]["embed"]) for q, cands, _, _ in part for c in cands]).float().view(len(part), 20)
        hard = torch.tensor(np.stack([l[3] for l in part]), device=DEVICE)
        teacher_logits = torch.tensor(np.stack([l[2] for l in part]), device=DEVICE)
        has = hard.sum(-1) > 0
        # a "wrong" candidate the teacher rates at 95% of the right one or more is probably right too: no "no" lesson for it
        best_right = torch.where(hard > 0, torch.sigmoid(teacher_logits), torch.zeros_like(hard)).max(-1, keepdim=True).values
        keep = ((hard > 0) | (torch.sigmoid(teacher_logits) < 0.95 * best_right) | ~has[:, None]).float()
        loss = (torch.nn.functional.binary_cross_entropy_with_logits(out, hard, reduction="none") * keep).sum() / keep.sum()
        if has.any():
            teach = torch.softmax(teacher_logits[has] / TEMP, dim=-1)
            target = 0.2 * teach + 0.8 * hard[has] / hard[has].sum(-1, keepdim=True)   # mostly the known right answer
            loss = loss + torch.nn.functional.kl_div(torch.log_softmax(out[has], -1), target, reduction="batchmean")
        loss.backward(); opt.step(); sched.step(); opt.zero_grad(); step += 1
        if step % 2000 == 0: print(f"step {step}/{steps} loss {loss.item():.3f} · {time.time() - started:.0f}s", end=" | ")
    saved.append({k: v.detach().clone().float().cpu() for k, v in student.state_dict().items()})   # each epoch's end
student.eval()
print(f"\ntrained in {time.time() - started:.0f}s")
# the model soup: the epochs' weights averaged; kept only if it is right more often on held-out wording
def unseen_right(model, take=3000):
    queries, wants = TESTS["unseen"]; near = ranked(st_encoder(SEARCH), queries[:take], k=10); ok = 0
    for i, (top, _) in enumerate(near):
        with torch.no_grad():
            p = student_logits(model, [(queries[i], docs[c]["embed"]) for c in top]).float().cpu().numpy()
        ok += right("unseen", top[int(np.argmax(p))], wants[i])
    return ok / take
last_score = unseen_right(student)
final = {k: v.clone() for k, v in student.state_dict().items()}
student.load_state_dict({k: (sum(s[k] for s in saved) / len(saved)).to(final[k].dtype) for k in final})
soup_score = unseen_right(student)
if soup_score < last_score:
    student.load_state_dict(final)
results["student last epoch vs soup (unseen, top 10)"] = [last_score, soup_score]
print(f"held-out wording: last epoch {last_score:.1%} · soup {soup_score:.1%} → keeping the {'soup' if soup_score >= last_score else 'last epoch'}")

In [ ]:
# 7. How good, how deep, how sure: the student reordering the nearest 5, 10 or 20; then on held-out wording the lead it
# needs to be right 99% of the time (the gate) and the score under which a question is "not in the surveys" (the floor)
def student_probs(pairs):
    with torch.no_grad():
        return torch.sigmoid(student_logits(student, pairs).float()).cpu().numpy()
scored = {name: [student_probs([(queries[i], docs[c]["embed"]) for c in NEAREST[name][i][0]]) for i in range(len(queries))]
          for name, (queries, _) in TESTS.items()}
def pick(name, i, depth):
    top, p = NEAREST[name][i][0][:depth], scored[name][i][:depth]
    order = np.argsort(-p); best = top[order[0]]
    other = next((p[j] for j in order[1:] if docs[top[j]]["group"] != docs[best]["group"]), 0.0)
    return best, float(p[order[0]]), float(p[order[0]] - other)
for depth in (5, 10, 20):
    for name, (queries, wants) in TESTS.items():
        results[f"student {name} top {depth}"] = float(np.mean([right(name, pick(name, i, depth)[0], wants[i]) for i in range(len(queries))]))
    print(f"student reading the nearest {depth}: unseen {results[f'student unseen top {depth}']:.1%} · golden {results[f'student golden top {depth}']:.1%}")
best = max(results[f"student golden top {d}"] for d in (5, 10, 20))
DEPTH = next(d for d in (5, 10, 20) if results[f"student golden top {d}"] >= best - 0.005)   # the shallowest within half a point: faster
rows = {name: np.array([(pick(name, i, DEPTH)[2], right(name, pick(name, i, DEPTH)[0], wants[i]), pick(name, i, DEPTH)[1])
                        for i in range(len(queries))]) for name, (queries, wants) in TESTS.items()}
u = rows["unseen"]
GATE = next((g for g in np.arange(0.0, 1.0, 0.01) if (u[:, 0] >= g).sum() >= 50 and u[u[:, 0] >= g, 1].mean() >= 0.99), 0.99)
for name, r in rows.items():
    kept = r[:, 0] >= GATE
    results[f"gate {name}"] = {"answered at once": float(kept.mean()), "right among those": float(r[kept, 1].mean()) if kept.any() else 0.0}
    print(f"gate {GATE:.2f}, {name}: answered at once {kept.mean():.0%}, right among those {results[f'gate {name}']['right among those']:.1%}")
off = np.array([student_probs([(q, docs[c]["embed"]) for c in top[:DEPTH]]).max() for (top, _), q in zip(ranked(st_encoder(SEARCH), OFF_TEST), OFF_TEST)])
real = rows["golden"][:, 2]
FLOOR = float(np.quantile(real, 0.02))   # 98% of real golden questions score above it
results["floor"] = {"floor": FLOOR, "off-topic caught": float((off < FLOOR).mean()), "real kept": float((real >= FLOOR).mean())}
print(f"depth {DEPTH} · gate {GATE:.2f} · floor {FLOOR:.3f}: off-topic said 'not in the surveys' {(off < FLOOR).mean():.0%}, real questions kept {(real >= FLOOR).mean():.0%}")

In [ ]:
# 8. Shrink both to the server's format (8-bit ONNX), check them with the server's own code path (api/ask_model.py), and
# time the reranker on one CPU thread at the depth chosen
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
OUT_SEARCH = f"ask-v5/{SEARCH_NAME}-int8"
os.makedirs("ask-v5/ask-reranker", exist_ok=True); os.makedirs(OUT_SEARCH, exist_ok=True)
names = ["input_ids", "attention_mask", "token_type_ids"]
def export(model, dummy, out_names, path):
    torch.onnx.export(model.cpu().eval(), tuple(dummy[k] for k in names), "full.onnx", input_names=names, output_names=out_names,
                      dynamic_axes={k: {0: "batch", 1: "tokens"} for k in names} | {out_names[0]: {0: "batch"}}, opset_version=17, dynamo=False)
    quantize_dynamic("full.onnx", path, weight_type=QuantType.QInt8)
options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1
export(student, stok(["a question"], ["a survey question"], return_tensors="pt"), ["logits"], "ask-v5/ask-reranker/model_optimized.onnx")
stok.save_pretrained("tok_r"); shutil.copy("tok_r/tokenizer.json", "ask-v5/ask-reranker/tokenizer.json")
rw = tokenizers.Tokenizer.from_file("ask-v5/ask-reranker/tokenizer.json"); rw.enable_truncation(256); rw.enable_padding(pad_id=0, pad_token="[PAD]")
rs = onnxruntime.InferenceSession("ask-v5/ask-reranker/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
def server_rerank(question, texts):   # api/ask_model.py rerank, line for line
    coded = rw.encode_batch([(question, t) for t in texts]); ids = np.array([c.ids for c in coded], np.int64)
    feed = {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64), "token_type_ids": np.array([c.type_ids for c in coded], np.int64)}
    return 1 / (1 + np.exp(-rs.run(None, feed)[0].reshape(-1)))
queries, wants = TESTS["golden"]; ok = 0; started = time.time()
for i, q in enumerate(queries):
    top = NEAREST["golden"][i][0][:DEPTH]
    ok += right("golden", top[int(np.argmax(server_rerank(q, [docs[c]["embed"] for c in top])))], wants[i])
results["reranker 8-bit golden"] = ok / len(queries)
results["reranker ms on one CPU thread"] = (time.time() - started) / len(queries) * 1000
print(f"reranker 8-bit, server code: golden {ok / len(queries):.1%} · {results['reranker ms on one CPU thread']:.0f} ms for {DEPTH} candidates on one CPU thread")
bert = SEARCH[0].auto_model
export(bert, SEARCH.tokenize(["a question"]), ["last_hidden_state"], f"{OUT_SEARCH}/model_optimized.onnx")
SEARCH.tokenizer.save_pretrained("tok_b"); shutil.copy("tok_b/tokenizer.json", f"{OUT_SEARCH}/tokenizer.json")
bw = tokenizers.Tokenizer.from_file(f"{OUT_SEARCH}/tokenizer.json"); bw.enable_truncation(512); bw.enable_padding(pad_id=0, pad_token="[PAD]")
bs = onnxruntime.InferenceSession(f"{OUT_SEARCH}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
def server_embed(texts):   # api/ask_model.py embed
    out = np.zeros((len(texts), 384), np.float32)
    for s in range(0, len(texts), 32):
        coded = bw.encode_batch(texts[s:s + 32]); ids = np.array([c.ids for c in coded], np.int64)
        out[s:s + 32] = bs.run(None, {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64), "token_type_ids": np.zeros_like(ids)})[0][:, 0]
    return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)
SEARCH.to(DEVICE)
results["search 8-bit"] = found_at(f"{SEARCH_NAME} 8-bit", server_embed)
closeness = np.array([c[0] for _, c in ranked(server_embed, [g["q"] for g in golden], k=1)])
near_floor = 0.64 if SEARCH_NAME == "bge-small-en-v1.5" else round(float(np.quantile(closeness, 0.02)), 3)   # today's stays as tuned
json.dump({"gate": round(float(GATE), 3), "floor": round(FLOOR, 4), "top": DEPTH}, open("ask-v5/ask-reranker/calibration.json", "w"))
json.dump({"near_floor": near_floor, "gate": 0.04}, open(f"{OUT_SEARCH}/calibration.json", "w"))
print("file sizes:", {f: f"{os.path.getsize(f) / 1e6:.0f} MB" for f in ["ask-v5/ask-reranker/model_optimized.onnx", f"{OUT_SEARCH}/model_optimized.onnx"]})

In [ ]:
json.dump(results, open("ask-v5/results.json", "w"), indent=1, default=str)
json.dump(qwen["phrases"], open("ask-v5/phrases.json", "w"), indent=0)
shutil.copy("qwen-v5.json", "ask-v5/")
print("WHAT TO SEND BACK: ask-v5.zip (it holds everything below)\n")
print(f"search, right one in the nearest 20 — today: golden {results['search today']['golden'][20]:.1%}, unseen {results['search today']['unseen'][20]:.1%}"
      f" · shipped ({SEARCH_NAME}, 8-bit): golden {results['search 8-bit']['golden'][20]:.1%}, unseen {results['search 8-bit']['unseen'][20]:.1%}")
print(f"teacher reordering 20: golden {results['teacher golden']:.1%} · student 8-bit at depth {DEPTH}: golden {results['reranker 8-bit golden']:.1%}, "
      f"{results['reranker ms on one CPU thread']:.0f} ms a question on one CPU thread")
print(f"gate {GATE:.2f}: golden answered at once {results['gate golden']['answered at once']:.0%}, right among those {results['gate golden']['right among those']:.1%}")
print(f"not in the surveys: {results['floor']['off-topic caught']:.0%} of off-topic caught, {results['floor']['real kept']:.0%} of real kept")
print("\nThe bar inside Ask (tested on the laptop after this zip comes back): today answers 85 of 161 golden questions at once,"
      " 98.8% of them right, and 155 of 161 right after the AI check (gpt-4.1). v5 ships only if it beats both.")
shutil.make_archive("ask-v5", "zip", "ask-v5")
print("\nREADY: ask-v5.zip is downloading to your Downloads folder.")
print("If the download does not start: left bar → folder icon → content → ask-v5.zip → ⋮ → Download (within 15 minutes).")
files.download("ask-v5.zip")

In [ ]:
# Stop paying once the download has had plenty of time: an idle GPU still uses units
import time
from google.colab import runtime
print("Releasing the GPU in 15 minutes. If ask-v5.zip is not in Downloads yet: folder icon → content → ask-v5.zip → ⋮ → Download")
time.sleep(900)
runtime.unassign()